In [1]:
!pip install transformers datasets accelerate sentencepiece pandas scikit-learn huggingface_hub

In [2]:
import torch
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [3]:
if torch.cuda.is_available():
    print("GPU (CUDA) is available and will be used for training.")
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    print(f"Current GPU device name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU (CUDA) is not available. Training will proceed on CPU.")

# The TrainingArguments object (defined in a previous cell) by default will use CUDA if available.
# No explicit modification to training_args is needed for default GPU usage.

GPU (CUDA) is available and will be used for training.
Number of GPUs available: 1
Current GPU device name: Tesla T4


In [4]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'\d+', ' number ', text)
    text = re.sub(r'[₹$]', ' currency ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [5]:
df = pd.read_csv("/content/Lables.csv")

df["Target"] = df["Target"].fillna("UNKNOWN")
df["Alias"] = df["Alias"].fillna("UNKNOWN")
df["Account"] = df["Account"].fillna("UNKNOWN")

df["text"] = df["Sender"] + " " + df["MessageContent"]
df["text"] = df["text"].apply(clean_text)

X = df["text"]

In [6]:
def train_model(label):

    y = df[label]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1,2),
            max_features=10000
        )),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    model.fit(X_train, y_train)

    return model

In [7]:
type_model = train_model("Type")
target_model = train_model("Target")
alias_model = train_model("Alias")
account_model = train_model("Account")

In [8]:
joblib.dump(type_model,"type_model.joblib")
joblib.dump(target_model,"target_model.joblib")
joblib.dump(alias_model,"alias_model.joblib")
joblib.dump(account_model,"account_model.joblib")

print("Models saved")

Models saved


In [9]:
test_text = "ICICI Bank Acct XX789 debited for Rs 80.00 on 26-Dec-25; ZOMATO credited"

test_clean = clean_text(test_text)

print("Type:", type_model.predict([test_clean])[0])
print("Target:", target_model.predict([test_clean])[0])
print("Alias:", alias_model.predict([test_clean])[0])
print("Account:", account_model.predict([test_clean])[0])

Type: Debit
Target: RAJESH REDDY MU
Alias: Unknown
Account: ICICI - XX789
